In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MyApplication").getOrCreate()

In [2]:
data = [("Alice", 100),("Bob", 89), ("Charlie", 45)]
cols = ["name","age"]
df = spark.createDataFrame(data, cols)
df.show()

+-------+---+
|   name|age|
+-------+---+
|  Alice|100|
|    Bob| 89|
|Charlie| 45|
+-------+---+



In [3]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)



In [4]:
rdd = spark.sparkContext.parallelize(data)
df = rdd.toDF(cols)
df

DataFrame[name: string, age: bigint]

In [5]:
df.show()

+-------+---+
|   name|age|
+-------+---+
|  Alice|100|
|    Bob| 89|
|Charlie| 45|
+-------+---+



In [6]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType(
    [
        StructField("name", StringType(), True), 
        StructField("age",IntegerType(), True)
    ]
)
df =spark.createDataFrame(data, schema)
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)



In [7]:
df.show()

+-------+---+
|   name|age|
+-------+---+
|  Alice|100|
|    Bob| 89|
|Charlie| 45|
+-------+---+



In [8]:
df.select("name").show()

+-------+
|   name|
+-------+
|  Alice|
|    Bob|
|Charlie|
+-------+



In [9]:
df.filter(df.age > 80).show()

+-----+---+
| name|age|
+-----+---+
|Alice|100|
|  Bob| 89|
+-----+---+



In [15]:
from pyspark.sql import functions as F
df.filter((F.col('age') > 80) & (F.col('name') == "Alice")).show()

+-----+---+
| name|age|
+-----+---+
|Alice|100|
+-----+---+



In [21]:
df_with_age = df.withColumn("age_bucket", 
                              F.when(F.col("age") < 35, "Youngster")
                                .when(F.col("age") < 60, "mid-citizen")
                                .otherwise("senior-citizen")).\
                                 withColumn("updated_age", F.col("age") + 5)

df_with_age

DataFrame[name: string, age: int, age_bucket: string, updated_age: int]

In [22]:
df_with_age.show()

+-------+---+--------------+-----------+
|   name|age|    age_bucket|updated_age|
+-------+---+--------------+-----------+
|  Alice|100|senior-citizen|        105|
|    Bob| 89|senior-citizen|         94|
|Charlie| 45|   mid-citizen|         50|
+-------+---+--------------+-----------+



In [ ]:
# df.column_name vs F.col

# 1. df['first name'], F.col("first name")
# 2. df1.join(df2, "id").filter(df["id"] > 10)

# df1.alias("a").join(df2.alias("b"), "id" ).filter(F.col("a.id") > 10)

# df.where("first name == 'John' ")

In [23]:
data = [("Alice", 100),("Bob", 89), ("Charlie", 45)]
cols = ["name","id"]
df1 = spark.createDataFrame(data, cols)
df1.show()

+-------+---+
|   name| id|
+-------+---+
|  Alice|100|
|    Bob| 89|
|Charlie| 45|
+-------+---+



In [27]:
df1.printSchema()

root
 |-- name: string (nullable = true)
 |-- id: long (nullable = true)



In [36]:
data = [("Maths", 100),("Science", 89), ("SocialScience", 45)]
cols = ["subject","user_id"]
df2 = spark.createDataFrame(data, cols)
df2.show()

+-------------+-------+
|      subject|user_id|
+-------------+-------+
|        Maths|    100|
|      Science|     89|
|SocialScience|     45|
+-------------+-------+



In [37]:
df2.printSchema()

root
 |-- subject: string (nullable = true)
 |-- user_id: long (nullable = true)



In [33]:
df1.join(df2, "id").filter(df1["id"] > 10).show()

+---+-------+-------------+
| id|   name|      subject|
+---+-------+-------------+
| 45|Charlie|SocialScience|
| 89|    Bob|      Science|
|100|  Alice|        Maths|
+---+-------+-------------+



In [34]:
df1.alias("a").join(df2.alias("b"), "id" ).filter(F.col("a.id") > 10).show()


+---+-------+-------------+
| id|   name|      subject|
+---+-------+-------------+
| 45|Charlie|SocialScience|
| 89|    Bob|      Science|
|100|  Alice|        Maths|
+---+-------+-------------+



In [45]:
df1.join(df2, [df1.id == df2.user_id], 'inner').show()


+-------+---+-------------+-------+
|   name| id|      subject|user_id|
+-------+---+-------------+-------+
|Charlie| 45|SocialScience|     45|
|    Bob| 89|      Science|     89|
|  Alice|100|        Maths|    100|
+-------+---+-------------+-------+



In [40]:
help(df1.join)

Help on method join in module pyspark.sql.dataframe:

join(other: 'DataFrame', on: Union[str, List[str], pyspark.sql.column.Column, List[pyspark.sql.column.Column], NoneType] = None, how: Optional[str] = None) -> 'DataFrame' method of pyspark.sql.dataframe.DataFrame instance
    Joins with another :class:`DataFrame`, using the given join expression.
    
    .. versionadded:: 1.3.0
    
    .. versionchanged:: 3.4.0
        Supports Spark Connect.
    
    Parameters
    ----------
    other : :class:`DataFrame`
        Right side of the join
    on : str, list or :class:`Column`, optional
        a string for the join column name, a list of column names,
        a join expression (Column), or a list of Columns.
        If `on` is a string or a list of strings indicating the name of the join column(s),
        the column(s) must exist on both sides, and this performs an equi-join.
    how : str, optional
        default ``inner``. Must be one of: ``inner``, ``cross``, ``outer``,
      

In [46]:
df = spark.read.csv("data/employee_data (1).csv", header = True, inferSchema=True)
df.show()

+-------+---+------+----------+
|   name|age|salary|department|
+-------+---+------+----------+
|  Alice| 25|  5000|        HR|
|    Bob| 30|  6000|        IT|
|Charlie| 35|  7000|   Finance|
|  David| 28|  5500|     Sales|
|    Eve| 32|  6200|        IT|
|  Frank| 40|  7500|   Finance|
|  Grace| 27|  5300|        HR|
|   Hank| 33|  6400|        IT|
|    Ivy| 29|  5700|     Sales|
|   Jack| 38|  7200|   Finance|
|  Kelly| 26|  5100|        HR|
|   Liam| 31|  5900|        IT|
|    Mia| 34|  6800|   Finance|
|   Noah| 27|  5400|     Sales|
| Olivia| 36|  7100|        IT|
|  Peter| 29|  5600|        HR|
|  Quinn| 37|  7300|   Finance|
|   Rose| 28|  5500|     Sales|
|    Sam| 33|  6500|        IT|
|   Tina| 35|  7000|   Finance|
+-------+---+------+----------+
only showing top 20 rows



In [47]:
kpi1 = df.groupby("department").agg(F.sum("salary").alias("total_dept_salary"))
kpi1.show()

+----------+-----------------+
|department|total_dept_salary|
+----------+-----------------+
|     Sales|            65000|
|        HR|            60400|
|   Finance|            86400|
|        IT|            91100|
+----------+-----------------+



In [49]:
kpi2 = df.groupby("department").agg(F.count("*").alias("Total_std"), F.avg("salary").alias("Avg salary"), F.expr("percentile(salary, 0.25)").alias("quarter salary"))
kpi2.show()

+----------+---------+-----------------+--------------+
|department|Total_std|       Avg salary|quarter salary|
+----------+---------+-----------------+--------------+
|     Sales|       11|5909.090909090909|        5350.0|
|        HR|       11|5490.909090909091|        5100.0|
|   Finance|       13|6646.153846153846|        6200.0|
|        IT|       14|6507.142857142857|        6025.0|
+----------+---------+-----------------+--------------+



In [50]:
# broadcasting

df1 = spark.createDataFrame([(1, "alice"), (2, "bob"), (3, "cathy"), (4, "david")], ["id","name"])

df1.show()

+---+-----+
| id| name|
+---+-----+
|  1|alice|
|  2|  bob|
|  3|cathy|
|  4|david|
+---+-----+



In [51]:
df2 = spark.createDataFrame([(1,"HR"),(2,"IT")], ["id","dept"])
df2.show()

+---+----+
| id|dept|
+---+----+
|  1|  HR|
|  2|  IT|
+---+----+



In [52]:
df_join = df1.join(df2, "id", "inner")
df_join.show()

+---+-----+----+
| id| name|dept|
+---+-----+----+
|  1|alice|  HR|
|  2|  bob|  IT|
+---+-----+----+



In [53]:
df_join.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id#423L, name#424, dept#437]
   +- SortMergeJoin [id#423L], [id#436L], Inner
      :- Sort [id#423L ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(id#423L, 200), ENSURE_REQUIREMENTS, [plan_id=1075]
      :     +- Filter isnotnull(id#423L)
      :        +- Scan ExistingRDD[id#423L,name#424]
      +- Sort [id#436L ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(id#436L, 200), ENSURE_REQUIREMENTS, [plan_id=1076]
            +- Filter isnotnull(id#436L)
               +- Scan ExistingRDD[id#436L,dept#437]




In [ ]:
# 1. shuffling both the dataframes across nodes (id)

# 2. sorting within partition

# 3. merge phase (merge sort)

# works well for both the tables are big, data cannot fit in memory

In [54]:
# df1 is small and df2 is big

df_broadcast = df1.join(F.broadcast(df2), "id", "inner")
df_broadcast.show()

+---+-----+----+
| id| name|dept|
+---+-----+----+
|  1|alice|  HR|
|  2|  bob|  IT|
+---+-----+----+



In [55]:
df_broadcast.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id#423L, name#424, dept#437]
   +- BroadcastHashJoin [id#423L], [id#436L], Inner, BuildRight, false
      :- Filter isnotnull(id#423L)
      :  +- Scan ExistingRDD[id#423L,name#424]
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=1168]
         +- Filter isnotnull(id#436L)
            +- Scan ExistingRDD[id#436L,dept#437]




In [ ]:
# Initial stage

# Node 1 : df1 - (1,2,3)
# node 2 : df1 - (4,5)

# node 1 :df2 - (2,4)
# node 2: df2 - (1,5)

In [ ]:
# shuffle starts

# partition id = hash(id) % num of partitions in spark cluster

In [ ]:
# spark will move rows across nodes

# all rows with id =1 -> partition 1
# all rows with id =2  -> partition 2

# (disk write, network transfer, disk read)

In [ ]:
# align them properly
# partition 1:
# df1 -> (1,....)
# df2 -> (1, ....)

# partition 2:
# df1 -> (2, ...)
# df2 -> (2, ...)

# sorting inside each partition

# df1 sorted on the basis of id

# df2 sorted on the basis of id


# merge join

# spark scans both df1 and df2
# match rows efficiently like merge sort

In [75]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

spark = SparkSession.builder.getOrCreate()

# A typical messy dataset
data = [("Order_1", 100, 10), ("Order_2", 200, 20)]
columns = ["Order ID", "Price", "Tax"]

df = spark.createDataFrame(data, columns)
# df.Order ID

In [76]:
df.select(f.col("Order ID")).show()

+--------+
|Order ID|
+--------+
| Order_1|
| Order_2|
+--------+



In [77]:
df.select(df['Order ID']).show()

+--------+
|Order ID|
+--------+
| Order_1|
| Order_2|
+--------+



In [78]:
df.withColumn("Total", f.col("Price") + f.col("Tax")) \
  .withColumn("Discount", df.Total * 0.1) 

AttributeError: 'DataFrame' object has no attribute 'Total'

In [80]:
df.withColumn("Total", f.col("Price") + f.col("Tax")) \
  .withColumn("Discount", f.col("Total") * 0.1).show()

+--------+-----+---+-----+--------+
|Order ID|Price|Tax|Total|Discount|
+--------+-----+---+-----+--------+
| Order_1|  100| 10|  110|    11.0|
| Order_2|  200| 20|  220|    22.0|
+--------+-----+---+-----+--------+



In [81]:
df = spark.read.csv("data/employee_data (1).csv", header = True, inferSchema=True)
df.show()

+-------+---+------+----------+
|   name|age|salary|department|
+-------+---+------+----------+
|  Alice| 25|  5000|        HR|
|    Bob| 30|  6000|        IT|
|Charlie| 35|  7000|   Finance|
|  David| 28|  5500|     Sales|
|    Eve| 32|  6200|        IT|
|  Frank| 40|  7500|   Finance|
|  Grace| 27|  5300|        HR|
|   Hank| 33|  6400|        IT|
|    Ivy| 29|  5700|     Sales|
|   Jack| 38|  7200|   Finance|
|  Kelly| 26|  5100|        HR|
|   Liam| 31|  5900|        IT|
|    Mia| 34|  6800|   Finance|
|   Noah| 27|  5400|     Sales|
| Olivia| 36|  7100|        IT|
|  Peter| 29|  5600|        HR|
|  Quinn| 37|  7300|   Finance|
|   Rose| 28|  5500|     Sales|
|    Sam| 33|  6500|        IT|
|   Tina| 35|  7000|   Finance|
+-------+---+------+----------+
only showing top 20 rows



In [82]:
from pyspark.sql.window import Window

window_obj = Window.partitionBy("department").orderBy(F.col("salary").desc())

ranked = df.select("name", "department","salary", F.dense_rank().over(window_obj).alias('dense rank'))
ranked.show()

+-------+----------+------+----------+
|   name|department|salary|dense rank|
+-------+----------+------+----------+
|  Frank|   Finance|  7500|         1|
|  Wendy|   Finance|  7400|         2|
|  Quinn|   Finance|  7300|         3|
|   Jack|   Finance|  7200|         4|
|Charlie|   Finance|  7000|         5|
|   Tina|   Finance|  7000|         5|
|    Mia|   Finance|  6800|         6|
|  Ethan|   Finance|  6400|         7|
| Quincy|   Finance|  6400|         7|
|  Isaac|   Finance|  6200|         8|
|Ulysses|   Finance|  6200|         8|
|  Aaron|   Finance|  5500|         9|
|  Mason|   Finance|  5500|         9|
|  Laura|        HR|  6800|         1|
|   Yara|        HR|  6000|         2|
|  Peter|        HR|  5600|         3|
|  Daisy|        HR|  5600|         3|
|  Paula|        HR|  5600|         3|
|  Grace|        HR|  5300|         4|
|    Uma|        HR|  5200|         5|
+-------+----------+------+----------+
only showing top 20 rows



In [83]:
ranked.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (6)
+- Window (5)
   +- Sort (4)
      +- Exchange (3)
         +- Project (2)
            +- Scan csv  (1)


(1) Scan csv 
Output [3]: [name#605, salary#607, department#608]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/work/data/employee_data (1).csv]
ReadSchema: struct<name:string,salary:int,department:string>

(2) Project
Output [3]: [name#605, department#608, salary#607]
Input [3]: [name#605, salary#607, department#608]

(3) Exchange
Input [3]: [name#605, department#608, salary#607]
Arguments: hashpartitioning(department#608, 200), ENSURE_REQUIREMENTS, [plan_id=1335]

(4) Sort
Input [3]: [name#605, department#608, salary#607]
Arguments: [department#608 ASC NULLS FIRST, salary#607 DESC NULLS LAST], false, 0

(5) Window
Input [3]: [name#605, department#608, salary#607]
Arguments: [dense_rank(salary#607) windowspecdefinition(department#608, salary#607 DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentro

In [84]:
ranked.printSchema()

root
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- dense rank: integer (nullable = false)



In [88]:
# add salary increments

@F.udf("float") #return data type
def increment_salary(x):
    return x + (x * 0.10)

df_udf = ranked.withColumn("increment_salary", increment_salary(F.col("salary")))
df_udf.show()

+-------+----------+------+----------+----------------+
|   name|department|salary|dense rank|increment_salary|
+-------+----------+------+----------+----------------+
|  Frank|   Finance|  7500|         1|          8250.0|
|  Wendy|   Finance|  7400|         2|          8140.0|
|  Quinn|   Finance|  7300|         3|          8030.0|
|   Jack|   Finance|  7200|         4|          7920.0|
|Charlie|   Finance|  7000|         5|          7700.0|
|   Tina|   Finance|  7000|         5|          7700.0|
|    Mia|   Finance|  6800|         6|          7480.0|
|  Ethan|   Finance|  6400|         7|          7040.0|
| Quincy|   Finance|  6400|         7|          7040.0|
|  Isaac|   Finance|  6200|         8|          6820.0|
|Ulysses|   Finance|  6200|         8|          6820.0|
|  Aaron|   Finance|  5500|         9|          6050.0|
|  Mason|   Finance|  5500|         9|          6050.0|
|  Laura|        HR|  6800|         1|          7480.0|
|   Yara|        HR|  6000|         2|          

In [1]:
from pyspark.sql import SparkSession, functions as F

# --------------------------------------------------
# 1️⃣ Spark setup
# --------------------------------------------------
spark = SparkSession.builder \
    .appName("Skew_vs_Salting_Demo") \
    .getOrCreate()

# Reduce partitions to exaggerate skew
spark.conf.set("spark.sql.shuffle.partitions", 5)

# --------------------------------------------------
# 2️⃣ Create skewed dataset
# --------------------------------------------------
data = [("A", i) for i in range(900000)] + \
       [("B", i) for i in range(50000)] + \
       [("C", i) for i in range(50000)]

df = spark.createDataFrame(data, ["key", "value"])

print("Total rows:", df.count())


Total rows: 1000000


In [2]:
# --------------------------------------------------
# 3️⃣ WITHOUT SALTING (Skew problem)
# --------------------------------------------------
print("\n=== WITHOUT SALTING ===")

df_grouped = df.groupBy("key").count()

print("Final result:")
df_grouped.show()

print("Partition distribution AFTER shuffle:")
df_grouped.withColumn("pid", F.spark_partition_id()) \
    .groupBy("pid") \
    .count() \
    .show()




=== WITHOUT SALTING ===
Final result:
+---+------+
|key| count|
+---+------+
|  A|900000|
|  B| 50000|
|  C| 50000|
+---+------+

Partition distribution AFTER shuffle:
+---+-----+
|pid|count|
+---+-----+
|  0|    3|
+---+-----+



In [3]:
# --------------------------------------------------
# 4️⃣ ADD SALT COLUMN
# --------------------------------------------------
print("\n=== ADDING SALT ===")

salt_buckets = 10

df_salted = (
    df
    .withColumn("salt", (F.rand() * salt_buckets).cast("int"))
    .withColumn("salted_key", F.concat_ws("_", "key", "salt"))
)

print("Sample salted rows:")
df_salted.select("key", "value", "salt", "salted_key").show(10, False)


=== ADDING SALT ===
Sample salted rows:
+---+-----+----+----------+
|key|value|salt|salted_key|
+---+-----+----+----------+
|A  |0    |0   |A_0       |
|A  |1    |7   |A_7       |
|A  |2    |3   |A_3       |
|A  |3    |5   |A_5       |
|A  |4    |5   |A_5       |
|A  |5    |6   |A_6       |
|A  |6    |6   |A_6       |
|A  |7    |5   |A_5       |
|A  |8    |9   |A_9       |
|A  |9    |5   |A_5       |
+---+-----+----+----------+
only showing top 10 rows



In [4]:
# --------------------------------------------------
# 5️⃣ PARTIAL AGGREGATION (Distributed)
# --------------------------------------------------
print("\n=== WITH SALTING (PARTIAL AGG) ===")

partial = df_salted.groupBy("salted_key").count()

print("Partition distribution AFTER salting:")
partial.withColumn("pid", F.spark_partition_id()) \
    .groupBy("pid") \
    .count() \
    .show()

print("Sample partial aggregation:")
partial.orderBy("salted_key").show(20, False)


=== WITH SALTING (PARTIAL AGG) ===
Partition distribution AFTER salting:
+---+-----+
|pid|count|
+---+-----+
|  0|   30|
+---+-----+

Sample partial aggregation:
+----------+-----+
|salted_key|count|
+----------+-----+
|A_0       |90186|
|A_1       |89967|
|A_2       |89494|
|A_3       |89778|
|A_4       |90160|
|A_5       |90037|
|A_6       |89657|
|A_7       |90283|
|A_8       |89835|
|A_9       |90603|
|B_0       |5053 |
|B_1       |5006 |
|B_2       |4929 |
|B_3       |4875 |
|B_4       |4992 |
|B_5       |5023 |
|B_6       |5066 |
|B_7       |5021 |
|B_8       |4973 |
|B_9       |5062 |
+----------+-----+
only showing top 20 rows



In [5]:
# --------------------------------------------------
# 6️⃣ FINAL AGGREGATION (Unsalt)
# --------------------------------------------------
print("\n=== FINAL AGGREGATION (UNSALTING) ===")

final = (
    partial
    .withColumn("key", F.split("salted_key", "_")[0])
    .groupBy("key")
    .agg(F.sum("count").alias("count"))
)

final.show()


=== FINAL AGGREGATION (UNSALTING) ===
+---+------+
|key| count|
+---+------+
|  A|900000|
|  B| 50000|
|  C| 50000|
+---+------+



In [6]:
# --------------------------------------------------
# 7️⃣ OPTIONAL: Compare partition workload directly
# --------------------------------------------------
print("\n=== PARTITION WORKLOAD COMPARISON ===")

def count_in_partition(iterator):
    yield sum(1 for _ in iterator)

print("Without salting workload:")
print(df.groupBy("key").count().rdd.mapPartitions(count_in_partition).collect())

print("With salting workload:")
print(partial.rdd.mapPartitions(count_in_partition).collect())


=== PARTITION WORKLOAD COMPARISON ===
Without salting workload:
[3]
With salting workload:
[30]


In [ ]:
# Without salting:
# One partition does most of the work (because of "A")
# Others are underutilized
# With salting:
# "A" split into A_0 ... A_9
# Work evenly distributed across partitions
# Final result still correct after unsalting

<!-- In this script, you first create a deliberately skewed dataset where one key ("A") has far more rows than others, which causes a problem during Spark’s groupBy because all rows with the same key are sent to a single partition, leading to uneven workload and poor performance. To address this, you apply a technique called salting, where you add a random suffix to the key (turning "A" into values like "A_0", "A_1", etc.), effectively spreading the heavy data across multiple partitions so the work can be processed in parallel more evenly. After this first distributed aggregation, you then remove the artificial salt by extracting the original key and aggregating again, which combines the partial results back into the correct final counts. This approach demonstrates how salting helps mitigate skew by redistributing data during processing while still preserving accurate results at the end. -->

In [14]:
help(spark.createDataFrame)

Help on method createDataFrame in module pyspark.sql.session:

createDataFrame(data: Union[pyspark.rdd.RDD[Any], Iterable[Any], ForwardRef('PandasDataFrameLike'), ForwardRef('ArrayLike')], schema: Union[pyspark.sql.types.AtomicType, pyspark.sql.types.StructType, str, NoneType] = None, samplingRatio: Optional[float] = None, verifySchema: bool = True) -> pyspark.sql.dataframe.DataFrame method of pyspark.sql.session.SparkSession instance
    Creates a :class:`DataFrame` from an :class:`RDD`, a list, a :class:`pandas.DataFrame`
    or a :class:`numpy.ndarray`.
    
    .. versionadded:: 2.0.0
    
    .. versionchanged:: 3.4.0
        Supports Spark Connect.
    
    Parameters
    ----------
    data : :class:`RDD` or iterable
        an RDD of any kind of SQL data representation (:class:`Row`,
        :class:`tuple`, ``int``, ``boolean``, etc.), or :class:`list`,
        :class:`pandas.DataFrame` or :class:`numpy.ndarray`.
    schema : :class:`pyspark.sql.types.DataType`, str or list, op